###Ejercicio 1.1: DolarAPI — request básico — SETUP

In [0]:
import requests
import json

# GET request a DolarAPI — retorna una lista de JSONs planos
response = requests.get("https://dolarapi.com/v1/dolares")

data = response.json()
print(f"Status code: {response.status_code}")

for cotizacion in data:
    print(f"{cotizacion['nombre']:>15}: compra={cotizacion['compra']}, venta={cotizacion['venta']}")

###Ejercicio 1.2: DolarAPI → Bronze Delta — GUIDED


In [0]:
from pyspark.sql.functions import current_timestamp

# Normalizar tipos numéricos: convertir compra y venta a float para evitar conflicto int/float
data_normalized = [
    {**record, 'compra': float(record['compra']), 'venta': float(record['venta'])}
    for record in data
]

# JSON plano → DataFrame directo con createDataFrame
df_dolar = spark.createDataFrame(data_normalized)

# Agregar timestamp de ingesta para saber cuándo se cargó
df_dolar_bronze = df_dolar.withColumn("ingesta_timestamp", current_timestamp())
df_dolar_bronze.write.format("delta").mode("overwrite").saveAsTable("bootcamp.bronze.cotizacion_dolar")

spark.table("bootcamp.bronze.cotizacion_dolar").show()

### Ejercicio 1.3 — INDEPENDENT

In [0]:
# OpenMeteo: JSON anidado — hay que desanidar hourly.time, hourly.temperature_2m, etc.
url = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude": -34.6037,
    "longitude": -58.3816,
    "hourly": "temperature_2m,wind_speed_10m",
    "forecast_days": 2
}
response = requests.get(url, params=params)
data_clima = response.json()

# Desanidar: recorrer las listas paralelas dentro de hourly
hourly = data_clima["hourly"]
registros = []
for i in range(len(hourly["time"])):
    registros.append({
        "timestamp": hourly["time"][i],
        "temperatura_c": hourly["temperature_2m"][i],
        "viento_kmh": hourly["wind_speed_10m"][i]
    })

df_clima = spark.createDataFrame(registros)
df_clima_bronze = df_clima.withColumn("ingesta_timestamp", current_timestamp())
df_clima_bronze.write.format("delta").mode("overwrite").saveAsTable("bootcamp.bronze.clima_api")

spark.table("bootcamp.bronze.clima_api").show(10)

###Ejercicio 1.4: RapidAPI — elegí tu API — INDEPENDENT


In [0]:
# GET request a RAPID para proyecto — retorna una lista de JSONs planos
import requests
import json

response = requests.get("https://booking-com15.p.rapidapi.com/api/v1/hotels/getNearbyCities?latitude=65.9667&longitude=-18.5333&languagecode=en-us")

print(f"Status code: {response.status_code}")

data_all = response.json()  # list de dicts (conexión S6)
print(f"Reservas: {len(data)}")

In [0]:
import requests
import json
from pyspark.sql.functions import current_timestamp

url = "https://booking-com15.p.rapidapi.com/api/v1/hotels/getNearbyCities?latitude=65.9667&longitude=-18.5333&languagecode=en-us"

params = {
    "dest_type": "city",
    "longitude": -18.53044,
    "latitude": 65.97109,
    "region": "North Iceland",
    "dest_id": -2643106,
    "name": "Dalvík",
    "nr_hotels": 11,
    "cc1": "is",
    "country": "Iceland"
}

response = requests.get(url, params=params)
data_booking = response.json()

# JSON plano → DataFrame directo con createDataFrame
df_booking = spark.createDataFrame(data_booking)

# Agregar timestamp de ingesta para saber cuándo se cargó
df_booking_bronze = df_booking.withColumn("ingesta_timestamp", current_timestamp())
df_booking_bronze.write.format("delta").mode("overwrite").saveAsTable("bootcamp.bronze.booking_api")

spark.table("bootcamp.bronze.booking_api").show(10)


